In [1]:
from parser import *

In [2]:
def get_test_case(pomdp, distance):
    goal_states, actions = pomdp.get_goal_states()
    bfs_distances = pomdp.get_bfs_distances(goal_states)
    max_dist = None
    pair_distance = None
    for v in bfs_distances[distance]:
        for v2 in bfs_distances[distance]:
            if v < v2:
                # print(v, v2, pomdp.get_state_similarity(v, v2))
                current = pomdp.get_state_similarity(v, v2)
                if max_dist is None or current > max_dist:
                    max_dist = current
                    pair_distance = (v, v2)
    print(pomdp.name, distance, "distance:", max_dist)
    return {pair_distance[0], pair_distance[1]}


In [3]:
for pomdp_name in ["cit.POMDP", "mit.POMDP", "pentagon.POMDP", "sunysb.POMDP"]:
    pomdp = POMDP(pomdp_name, is_navigation=True)

    for action in pomdp.actions:
        for v in pomdp.states:
            count = 0
            probs = set()
            for obs in pomdp.observations:
                prob = pomdp.get_obs_prob(action, v, obs)
                probs.add(prob)
            max_prob = max(probs)
            for obs in pomdp.observations:
                prob = pomdp.get_obs_prob(action, v, obs)
                if math.isclose(prob, max_prob, rel_tol=1e-5, abs_tol=1e-5):
                    count += 1
            if count != 1:
                print (pomdp_name, v, action, count, probs)

# Creating test cases

In [4]:
undet_obs = 27
def get_new_obs_f(pomdp):
    new_trans_f = {}
    for action in pomdp.actions:
        if action not in new_trans_f.keys():
            new_trans_f[action] = {}
        for v in pomdp.states:
            if v not in new_trans_f[action].keys():
                new_trans_f[action][v] = {}
            max_prob = None
            max_obs = None
            for obs in pomdp.observations:
                prob = pomdp.get_obs_prob(action, v, obs)
                if max_prob is None or prob > max_prob:
                    max_prob = prob
                    max_obs = obs

            assert(max_obs is not None)
            new_trans_f[action][v][max_obs] = max_prob
            new_trans_f[action][v][undet_obs] = 1 - max_prob

    return new_trans_f


In [5]:
for pomdp_name in ["cit.POMDP", "mit.POMDP", "pentagon.POMDP", "sunysb.POMDP"]:
    pomdp = POMDP(pomdp_name, is_navigation=True)
    testcase1 = get_test_case(pomdp, 1)
    testcase2 = get_test_case(pomdp, 2)
    testcase3 = get_test_case(pomdp, 3)
    new_obs_f = get_new_obs_f(pomdp)
    # pomdp.states = get_new_states(pomdp)
    pomdp.obs_f = new_obs_f
    pomdp.normalize()
    pomdp.write_test_case(testcase1, pomdp_name + "_1")
    pomdp.write_test_case(testcase2, pomdp_name + "_2")
    pomdp.write_test_case(testcase3, pomdp_name + "_3")

cit.POMDP 1 distance: 0.20055072393063572
cit.POMDP 2 distance: 0.20371616962623362
cit.POMDP 3 distance: 0.17477284686778782


/Users/stefaniemuroyalei/Documents/ist/probabilistic_hoare_triples/.env/lib/python3.13/site-packages/scipy/spatial/distance.py:1388: RuntimeWarning: invalid value encountered in sqrt
  return np.sqrt(js / 2.0)


mit.POMDP 1 distance: 0.1414781140615367
mit.POMDP 2 distance: 0.20067260495648714
mit.POMDP 3 distance: 0.20358866903540526
pentagon.POMDP 1 distance: 0.20055072393063572
pentagon.POMDP 2 distance: 0.20055072393063572
pentagon.POMDP 3 distance: 0.20371616962623362
sunysb.POMDP 1 distance: 0.18733122645673864
sunysb.POMDP 2 distance: 0.19535769692605823
sunysb.POMDP 3 distance: 0.20371616962623362


In [6]:
f = open("helper_robot_slurm.sh", "w")
for pomdp_name in ["cit.POMDP", "mit.POMDP", "pentagon.POMDP", "sunysb.POMDP"]:
    f.write(f"sbatch f1_slurm.sh {pomdp_name}_1\n")
    f.write(f"sbatch f1_slurm.sh {pomdp_name}_2\n")
    f.write(f"sbatch f1_slurm.sh {pomdp_name}_3\n")

f.close()

In [ ]:
 0.2037